# Bradley–Terry Article Ranking

This notebook constructs continuous political-bias scores for news articles using **pairwise comparisons** and a **Bradley–Terry ranking model**.

Rather than assigning each article a fixed categorical label, the pipeline:

1. Matches article sources to AllSides source ratings
2. Generates random article pairs
3. Determines the relatively more right-leaning article in each pair
4. Fits a Bradley–Terry model over the pairwise outcomes
5. Exports a continuous score for each ranked article

These scores are used as regression targets in the downstream NLP model.


## 1. Imports and source ratings

In [ ]:
import random
import re
from difflib import get_close_matches

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [91]:
AS_scrape = pd.read_csv(
    "https://raw.githubusercontent.com/trevor-hou/BTNet/refs/heads/main/notebooks/csvs/Allsides_source_matching"
)

AS_scrape.head()


,Unnamed: 0,name,allsides_page,bias,rating
0,0,ABC News (Online),https://www.allsides.com/news-source/abc-news-...,Lean Left,-1.42
1,1,ACLU,https://www.allsides.com/news-source/aclu-medi...,Lean Left,NaN
2,2,AlterNet,https://www.allsides.com/news-source/alternet-...,Left,-4.50
3,3,Andrew Sullivan,https://www.allsides.com/news-source/andrew-su...,Center,NaN
4,4,Ann Coulter,https://www.allsides.com/news-source/ann-coult...,Right,NaN


## 2. Load the article dataset

The article corpus contains article text and source names. Source-level AllSides ratings are used as weak supervision for pairwise comparisons. Use articles scraped via 01_allsides_scraping.


In [ ]:
articles = pd.read_csv(
    "[insert scraped news articles].csv"
)

articles = articles.dropna().reset_index(drop=True)

print(f"Articles loaded: {len(articles):,}")
articles.head()


## 3. Match article source names to AllSides ratings

Source names differ slightly across datasets, so names are normalized first, followed by manual aliases and fuzzy matching where necessary.


In [ ]:
keys = list(articles["source"].unique())
values = list(AS_scrape["name"].unique())

def normalize(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("&", "and")
    s = s.replace("usa today", "usatoday")
    s = s.replace("msnbc", "ms now")
    s = s.replace("the blaze", "blaze media")
    s = s.replace("newsmax - opinion", "newsmax opinion")
    s = re.sub(r"\bonline news\b", "digital", s)
    s = re.sub(r"\bonline\b", "digital", s)
    s = re.sub(r"\bnews\b", "news", s)
    s = re.sub(r"\bopinion\b", "opinion", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

manual_map = {
    "fox news online news": "Fox News Digital",
    "cnn online news": "CNN Digital",
    "nbc news online": "NBC News Digital",
    "npr online news": None,
    "the blaze": "Blaze Media",
    "newsmax opinion": "Newsmax (Opinion)",
    "newsmax news": "Newsmax (News)",
    "national review": "National Review (News)",
    "wall street journal news": None,
    "wall street journal opinion": None,
    "washington post fact check": None,
    "associated press fact check": "Associated Press Fact Check",
    "abc news online": "ABC News (Online)",
    "cbs news online": "CBS News (Online)",
    "new york times news": "New York Times (News)",
    "new york times opinion": "New York Times (Opinion)",
    "new york post news": "New York Post (News)",
    "new york post opinion": "New York Post (Opinion)",
    "fox news opinion": "Fox News (Opinion)",
    "bbc fact check": "BBC Fact Check",
    "allsides staff": None,
    "guest writer": None,
    "guest writer right": None,
    "guest writer left": None,
    "guest writer center": None,
    "multiple writers lean left": None,
    "multiple writers lean right": None,
    "multiple writers center": None,
    "multiple writers left": None,
    "multiple writers mixed": None,
}

normalized_values = {normalize(v): v for v in values}
mapping = {}

for k in keys:
    nk = normalize(k)

    if nk in normalized_values:
        mapping[k] = normalized_values[nk]
        continue

    if nk in manual_map:
        mapping[k] = manual_map[nk]
        continue

    matches = get_close_matches(
        nk,
        normalized_values.keys(),
        n=1,
        cutoff=0.72
    )

    mapping[k] = normalized_values[matches[0]] if matches else None


In [ ]:
articles["source"] = articles["source"].map(mapping)

bias_lookup_bias = dict(zip(AS_scrape["name"], AS_scrape["bias"]))
bias_lookup_rating = dict(zip(AS_scrape["name"], AS_scrape["rating"]))
articles["bias"] = articles["source"].map(bias_lookup_bias)
articles["rating"] = articles["source"].map(bias_lookup_rating)


articles = articles.dropna().reset_index(drop=True)

print(f"Articles with matched AllSides ratings: {len(articles):,}")
articles[["source", "bias", "rating"]].head()

In [ ]:
articles

## 4. Generate pairwise comparisons

Each article is represented by its text, source, and categorical bias rating. Random pairs are generated to create a large set of relative comparisons.


In [ ]:
text_source_bias = list(
    zip(
        articles["content"],
        articles["source"],
        articles["bias"]
    )
)

def random_pairs(data, n_pairs):
    pairs = []

    for _ in range(n_pairs):
        a, b = random.sample(data, 2)
        if "error" not in a and "error" not in b:
          pairs.append((a, b))

    return pairs

random.seed(42)
pairs = random_pairs(text_source_bias, 200000)

print(f"Generated pairs: {len(pairs):,}")


## 5. Determine pairwise winners

Categorical AllSides ratings provide the first comparison. If both sources are in the same category, the numerical AllSides rating is used as a tie-breaker.


In [ ]:
def bias_to_score(bias):
    if bias == "Left":
        return -2
    elif bias == "Lean Left":
        return -1
    elif bias == "Center":
        return 0
    elif bias == "Lean Right":
        return 1
    elif bias == "Right":
        return 2
    else:
        return None


def rank_articles(bias1, bias2):
    if bias_to_score(bias1[2]) > bias_to_score(bias2[2]):
        return bias1[0]

    elif bias_to_score(bias2[2]) > bias_to_score(bias1[2]):
        return bias2[0]

    else:
        rating1 = AS_scrape[
            AS_scrape["name"] == mapping[bias1[1]]
        ]["rating"].iloc[0]

        rating2 = AS_scrape[
            AS_scrape["name"] == mapping[bias2[1]]
        ]["rating"].iloc[0]

        if rating1 > rating2:
            return bias1[0]
        elif rating2 > rating1:
            return bias2[0]
        else:
            return None


In [ ]:
pairs = [(a, b, rank_articles(a, b)) for a, b in pairs]
pairs = [p for p in pairs if p[2] is not None]

print(f"Usable pairwise outcomes: {len(pairs):,}")


In [ ]:
def win_lose(articles):
    if articles[0][0] == articles[2]:
        return (articles[2], articles[1][0])
    else:
        return (articles[2], articles[0][0])

winning_articles = [win_lose(p) for p in pairs]


## 6. Fit the Bradley–Terry model

`choix.ilsr_pairwise` estimates a latent score for each article from the winner/loser comparison graph.


In [ ]:
!pip install -q choix

In [ ]:
import choix

matches = winning_articles

actors = sorted(set(x for m in matches for x in m))
actor_to_id = {a: i for i, a in enumerate(actors)}
id_to_actor = {i: a for a, i in actor_to_id.items()}

data = [
    (actor_to_id[w], actor_to_id[l])
    for w, l in matches
]

params = choix.ilsr_pairwise(
    len(actors),
    data,
    alpha=0.1
)

scores = {
    id_to_actor[i]: params[i]
    for i in range(len(actors))
}

print("matches:", len(matches))
print("actors:", len(actors))
print("scores:", len(scores))


## 7. Inspect score distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(list(scores.values()), bins=30)
plt.xlabel("Bradley–Terry Score")
plt.ylabel("Articles")
plt.title("Distribution of Article Bias Scores")
plt.tight_layout()
plt.show()


## 8. Export scores

The output contains the article text and its inferred continuous Bradley–Terry score. This file is the input to the BERT regression notebook.


In [ ]:
score_df = pd.DataFrame({
    "text": list(scores.keys()),
    "bias_score": list(scores.values())
})

score_df.to_csv("article_bias_scores.csv", index=False)

print(f"Saved {len(score_df):,} ranked articles to article_bias_scores.csv")
score_df.head()
